# 1.6) Reproducible Data Pipelines

A result is only as trustworthy as the data behind it, and data that cannot be re-fetched, re-checked, and re-read identically is a reproducibility risk. This subchapter surveys the file formats you will meet, shows how to fetch a remote file once and verify it, and builds the idempotent cache → hash → skip pattern that makes a pipeline repeatable. The generated-code bug at the end is the classic shortcut: trusting that a file on disk is the file you wanted.

:::{admonition} Learning objectives
:class: tip
- Choose between CSV, parquet, netCDF, zarr, and GeoTIFF by the shape and scale of the data.
- Fetch a file over HTTP and understand why a bare download is not reproducible.
- Pin a file by its SHA256 with pooch so changes are caught, not silently absorbed.
- Implement an idempotent fetch: cache → hash → skip.
- Point pandas and xarray at the cached path.
:::

## A tour of formats

Match the format to the data. **CSV** is universal plain text, human-readable but untyped and bulky — fine for small tables and interchange. **parquet** is a columnar binary format: typed, compressed, and fast for large tables. **netCDF** is the standard single-file container for labelled n-dimensional arrays with metadata. **zarr** stores the same array model as a chunked *directory* (or cloud object store), built for parallel and out-of-core access. **GeoTIFF** holds a raster grid together with its coordinate reference system.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

rng = np.random.default_rng(0)
n = 10_000
big = pd.DataFrame({
    "time": pd.date_range("2024-01-01", periods=n, freq="h"),
    "temp_celsius": (10 + rng.normal(0, 5, n)).round(2),
    "station": rng.choice(["BAS", "LUG", "JFJ"], n),
})

big.to_csv("obs.csv", index=False)         # text, untyped
big.to_parquet("obs.parquet")              # columnar binary, typed, compressed

print("csv bytes:    ", Path("obs.csv").stat().st_size)
print("parquet bytes:", Path("obs.parquet").stat().st_size)
print("parquet keeps dtypes:", pd.read_parquet("obs.parquet").dtypes.to_dict())

csv bytes:     294209
parquet bytes: 116098
parquet keeps dtypes: {'time': dtype('<M8[us]'), 'temp_celsius': dtype('float64'), 'station': <StringDtype(na_value=nan)>}


In [2]:
import xarray as xr
import warnings
warnings.filterwarnings("ignore", message=".*Consolidated metadata.*")

ds = xr.Dataset(
    {"t2m": (("time", "lat", "lon"), rng.normal(5, 3, size=(12, 4, 6)))},
    coords={"time": pd.date_range("2024-01-01", periods=12, freq="MS"),
            "lat": np.linspace(46.0, 47.5, 4), "lon": np.linspace(6.5, 9.0, 6)},
)
ds.to_netcdf("field.nc")               # one self-describing file
ds.to_zarr("field.zarr", mode="w")     # a chunked directory store, cloud-friendly

print("netcdf is a file:    ", Path("field.nc").is_file())
print("zarr is a directory: ", Path("field.zarr").is_dir())
print("reopened netcdf shape:", xr.open_dataset("field.nc")["t2m"].shape)

netcdf is a file:     True
zarr is a directory:  True
reopened netcdf shape: (12, 4, 6)


:::{admonition} GeoTIFF: rasters with geography
:class: note
GeoTIFF stores a raster grid (satellite imagery, a digital elevation model) together with its coordinate reference system and geotransform. It is read with rioxarray, which returns a coordinate-aware DataArray:

```python
import rioxarray
dem = rioxarray.open_rasterio("elevation.tif")   # dims (band, y, x), carries a CRS
```

The geospatial raster stack (rioxarray, rasterio, GDAL) is covered with geospatial data in a later subchapter; GeoTIFF appears here only to complete the formats tour.
:::

## Fetching data over HTTP

`requests` performs the HTTP GET underneath any download. The essential pattern checks the status and writes the bytes:

```python
import requests
r = requests.get(url, timeout=30)
r.raise_for_status()                 # turn a 404/500 into an exception
Path("data.csv").write_bytes(r.content)
```

This works, but it re-downloads on every run and verifies nothing about *which* file arrived. Pinning and caching are the next step.

## pooch: download once, verify always

`pooch.retrieve` wraps the request, caches the file, and checks its hash against a value you pin. Calling it again returns the cached path with no download.

```python
import pooch
path = pooch.retrieve(
    url="https://example.org/data/temperature.csv",
    known_hash="sha256:1a2b3c...",      # pin the exact file
    path=pooch.os_cache("mlees"),        # OS-appropriate cache folder
)
data = pd.read_csv(path)
```

The pinned hash is what makes the pipeline reproducible: if the remote file ever changes, the check fails loudly instead of feeding you different data in silence. The call is shown rather than run so this book builds without network access; the cell below implements the same cache → hash → skip logic on a local file, using pooch's real hashing.

## The idempotent fetch: cache, hash, skip

A robust fetch is *idempotent* — calling it repeatedly does the least work and always yields the same verified file. The logic is: if a cached copy exists and its hash matches, use it; otherwise (re)download and verify. Here a local file stands in for the remote source so the logic runs offline.

In [3]:
import shutil
import pooch

source = Path("source_temperature.csv")          # stands in for a remote URL
pd.DataFrame({"date": pd.date_range("2024-06-01", periods=5, freq="D"),
              "temp_celsius": [18.2, 17.5, 19.1, 16.8, 18.0]}).to_csv(source, index=False)
known_hash = pooch.file_hash(source)              # real sha256, computed once
print("known sha256:", known_hash[:16], "...")

cache = Path("cache/temperature.csv")

def fetch_verified(expected_hash):
    cache.parent.mkdir(exist_ok=True)
    if cache.exists() and pooch.file_hash(cache) == expected_hash:
        print("cache hit:  hash matches, skip download")
        return cache
    print("cache miss: download and verify")
    shutil.copy(source, cache)                    # stands in for the network download
    if pooch.file_hash(cache) != expected_hash:
        raise ValueError("hash mismatch after download")
    return cache

fetch_verified(known_hash)   # first call: miss -> download
fetch_verified(known_hash)   # second call: hit -> skip

known sha256: 428eeaa65c3c10f6 ...
cache miss: download and verify
cache hit:  hash matches, skip download


PosixPath('cache/temperature.csv')

:::{admonition} Computational-thinking fundamental: pin your inputs
:class: important
A pipeline is reproducible only if its inputs are pinned. A filename says what a file is called; a cryptographic hash says what it *is*. Pinning the hash converts a silent failure mode — the remote file changed, your numbers changed, and nobody noticed — into a loud, early error. The same instinct runs through this whole course: make assumptions explicit and checkable (units in names, dimensions by label, and now data by hash) so that when something drifts, the code complains instead of lying.
:::

In [4]:
# point pandas at the verified cache path
data = pd.read_csv(cache, parse_dates=["date"])
print("shape:", data.shape, "| mean temp:", data["temp_celsius"].mean().round(2))

shape: (5, 2) | mean temp: 17.92


:::{admonition} Quick exercise: a one-line integrity check
:class: note
Using `pooch.file_hash`, write a boolean expression that is True only when the cached file `cache/temperature.csv` exists and matches `known_hash`.
:::

:::{admonition} Solution
:class: note dropdown
```python
from pathlib import Path
import pooch
ok = Path("cache/temperature.csv").exists() and pooch.file_hash("cache/temperature.csv") == known_hash
print(ok)
```
:::

## When generated code lies: trusting a file that is merely present

Asked to "download the data if it is not already there", an assistant checks only whether the file exists. But a file can exist and still be wrong — truncated by an interrupted download, or left over from an older version. Existence is not integrity.

In [5]:
def fetch_unverified(cache_path, source_path):
    # download only if the file is missing (as an assistant returned it)
    p = Path(cache_path)
    if not p.exists():
        shutil.copy(source_path, p)
    return pd.read_csv(p)

# a previous interrupted download left a TRUNCATED file in the cache
stale = Path("cache_b/temperature.csv")
stale.parent.mkdir(exist_ok=True)
stale.write_text("date,temp_celsius\n2024-06-01,18.2\n")   # 1 of 5 rows

result = fetch_unverified(stale, source)
print("rows returned:", len(result), "(should be 5)")

rows returned: 1 (should be 5)


:::{admonition} Diagnosis: existence is not integrity
:class: warning
The cache already held a one-row, truncated file, so the existence check passed and the function returned corrupt data with no error — every downstream statistic is now silently wrong. The fix is to verify the content hash: re-fetch whenever the file is missing *or* its hash does not match the pinned value, and refuse to proceed if it still does not match. This is exactly what `pooch.retrieve(url, known_hash=...)` does for a real download.
:::

In [6]:
def fetch_verified_data(cache_path, source_path, expected_hash):
    p = Path(cache_path)
    if not (p.exists() and pooch.file_hash(p) == expected_hash):
        shutil.copy(source_path, p)               # re-fetch on miss OR hash mismatch
    if pooch.file_hash(p) != expected_hash:
        raise ValueError("hash mismatch: refusing corrupted data")
    return pd.read_csv(p)

result = fetch_verified_data(stale, source, known_hash)
print("rows returned:", len(result), "(now correct)")

rows returned: 5 (now correct)


:::{admonition} Going deeper: multi-file registries
:class: seealso dropdown
For several files, `pooch.create` centralises the base URL and a registry of names and hashes, fetched by name.

```python
POOCH = pooch.create(
    path=pooch.os_cache("mlees"),
    base_url="https://example.org/data/",
    registry={
        "temperature.csv": "sha256:1a2b3c...",
        "discharge.nc": "sha256:4d5e6f...",
    },
)
path = POOCH.fetch("temperature.csv")   # cached + hash-verified
```
:::

:::{admonition} Going deeper: DOI-versioned data (Zenodo, figshare)
:class: seealso dropdown
pooch can fetch directly from a DOI, which points to an immutable, archived version of a dataset.

```python
doi = "doi:10.5281/zenodo.5739406"
path = pooch.retrieve(url=f"{doi}/wind_hourly.csv", known_hash="md5:cf059b...")
```

A DOI is the strongest pin available: the archive guarantees the bytes behind that identifier never change.
:::

:::{admonition} Going deeper: intake catalogs
:class: seealso dropdown
[intake](https://intake.readthedocs.io/) describes datasets in a YAML catalog so that code refers to a dataset by name rather than by path or URL, separating *what* data you want from *where* it lives.

```python
import intake
cat = intake.open_catalog("catalog.yml")
ds = cat["era5_t2m"].to_dask()
```

This keeps notebooks stable when storage locations change.
:::

:::{admonition} Going deeper: provenance and FAIR
:class: seealso dropdown
FAIR data is Findable, Accessible, Interoperable, and Reusable. In practice: deposit data in an archive that issues a DOI (Zenodo, figshare), record the exact version and hash used, and keep the fetch code alongside the analysis. Provenance — a clear trail from raw input to final figure — is what lets someone else (including future you) reproduce a result rather than approximate it.
:::

:::{admonition} Takeaways
:class: danger
- Pick the format for the job: CSV for small interchange, parquet for large tables, netCDF for single-file labelled arrays, zarr for chunked/cloud arrays, GeoTIFF for georeferenced rasters.
- A bare `requests.get` re-downloads and verifies nothing; caching and hashing fix both.
- Pin every input by hash (or DOI): a hash says what a file *is*, not just what it is called.
- An idempotent fetch is cache → hash → skip; re-fetch on a miss or a hash mismatch.
- Checking only that a file exists is the silent failure to avoid — a truncated or stale file passes that test and corrupts everything downstream.
:::

## Resources

- [pooch documentation](https://www.fatiando.org/pooch/latest/) — retrieving single files, registries, hashing, and DOI downloads.
- [Earth and Environmental Data Science — All About Data](https://earth-env-data-science.github.io/lectures/data.html) — Abernathey and Key on fetching remote data with pooch, Zenodo DOIs, and FAIR practice.
:::